# Préparation et Audit des données

Ici, je fait l'audit des données brutes afin de préparer le terrain pour le pré-traitement. Ce que j'y fait : 
1. Identifier les images corrompues.
2. Inventorier les extensions non standards (JPG, jpeg, png, etc.) à uniformiser.
3. Repérer les 5 images parasites (`x_Removed_from...`) pour s'assurer de leur exclusion.

In [1]:
from pathlib import Path
from PIL import Image

PROJECT_ROOT = Path(".").resolve().parent if Path(".").resolve().name == "notebooks" else Path(".").resolve()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
LOGS_DIR = PROJECT_ROOT / "logs" / "preprocessing"

LOGS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Audit d'Intégrité : Détection d'Images Corrompues

In [2]:
def verify_images(directory: Path):
    corrupt_images = []
    valid_extensions = set()
    total_images = 0
    
    for file_path in directory.rglob('*'):
        if file_path.is_file():

            #if file.startswith('.'):
            #    continue
                
            total_images += 1
            valid_extensions.add(file_path.suffix)
            
            try:
                # Vérification de l'image
                with Image.open(file_path) as img:
                    img.verify()
            except Exception as e:
                corrupt_images.append(str(file_path.relative_to(PROJECT_ROOT)))
                
    return corrupt_images, valid_extensions, total_images

In [3]:
corrupt_imgs, extensions, total = verify_images(RAW_DATA_DIR)

print(f"Audit terminé : {total} images trouvées.")
print(f"Extensions rencontrées : {extensions}")
print(f"Images corrompues détectées : {len(corrupt_imgs)}")

if corrupt_imgs:
    log_path = LOGS_DIR / "corrupted_images.txt"
    with open(log_path, "w") as f:
        f.write("\n".join(corrupt_imgs))
    print("-> Liste sauvegardée.")

Audit terminé : 14087 images trouvées.
Extensions rencontrées : {'.jpg', '.JPG', '.png'}
Images corrompues détectées : 27
-> Liste sauvegardée.


## 2. Détection des 5 images parasites (Maize Healthy)
Les travaux de [gabrieldgf4](https://github.com/gabrieldgf4/PlantVillage-Dataset) ont détecté dans le [dataset initial](https://github.com/spMohanty/PlantVillage-Dataset) des images non `healthy` présentées comme telles. <br> 
>Je vais donc les exclure. Je m'assure d'abord de leur présence.

In [4]:
removed_dir = RAW_DATA_DIR / "x_Removed_from_Healthy_leaves"
removed_files = {f.name for f in removed_dir.iterdir() if f.is_file()}
plantvillage_maize_dir = RAW_DATA_DIR / "PlantVillage" / "Maize" /"Corn_(maize)___healthy"

if removed_dir.exists():
    for file in plantvillage_maize_dir.iterdir():
        if file.is_file() and file.name in removed_files:
            print(f"{file} est marqué pour suppression.")

**Elles sont absentes.** <br>
*[spMohanty](https://github.com/spMohanty/PlantVillage-Dataset) les a probablement enlevé lors des derniers commit sans mentionner clairement cette action.*